In [ ]:
from qbraid.transpiler import Conversion, ConversionGraph, transpile

The qBraid SDK does not support direct conversions between all supported packages. Instead, some conversions, like braket to qiskit, require multiple conversions within the default graph model, e.g. braket $\rightarrow$ qasm3 $\rightarrow$ qiskit.

In [ ]:
graph = ConversionGraph()

graph.plot()

While qBraid's default conversion graph handles the Braket ↔ Qiskit conversion via an OpenQASM 3 intermediate, you can also register a **custom direct conversion** into the graph using the `Conversion` API. When converting from Qiskit to Braket, the transpiler will use the shortest available path — and if the direct conversion fails, it will automatically fall back to the native qBraid path.

In [ ]:
import logging

logger = logging.getLogger()

logger.setLevel(logging.INFO)

In [ ]:
from qiskit import QuantumCircuit

qiskit_circuit = QuantumCircuit(2)

qiskit_circuit.h(0)
qiskit_circuit.cx(0, 1)

qiskit_circuit.draw()

In [ ]:
braket_circuit = transpile(qiskit_circuit, "braket", conversion_graph=graph)

print(braket_circuit)

And now, we can try the same example, but with the direct conversion as a new shortest path

In [ ]:
from qbraid import transpile as _transpile


def qiskit_to_braket_direct(circuit):
    """Custom direct conversion via OpenQASM 3 intermediate."""
    qasm_str = _transpile(circuit, "qasm3")
    return _transpile(qasm_str, "braket")

In [ ]:
conversion = Conversion("qiskit", "braket", qiskit_to_braket_direct)

In [ ]:
try:
    graph.add_conversion(conversion, overwrite=True)
except Exception:
    # Workaround for SDK graph edge inconsistency:
    # remove stale _conversions entry and add fresh
    graph._conversions = [
        c for c in graph._conversions if not (c.source == "qiskit" and c.target == "braket")
    ]
    graph.add_conversion(conversion)

In [ ]:
graph.plot()

In [ ]:
braket_circuit = transpile(qiskit_circuit, "braket", conversion_graph=graph)

print(braket_circuit)

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>